# Prepare and stage the fraud data

[Guide](README.md) · [00: prepare data](00_prepare_data.ipynb) · [01: experiment](01_experiment.ipynb) · [02: results](02_results.ipynb) · [03: predictor](03_predictor.ipynb)

Read the [Connection setup](README.md#before-opening-the-notebooks) and fill in `.env` before uploading. First choose and inspect your local CSV files below; only the **Upload** cell contacts S3. This sample expects the seven fraud-detection features and the binary label `fraud`.

In [ ]:
from pathlib import Path
requirements = Path('requirements.txt') if Path('workflow.py').exists() else Path('automl/requirements.txt')
%pip install -r {requirements}

Restart the kernel if installation requests it, then continue below.

In [2]:
from pathlib import Path
import sys

# Works when Jupyter starts in either the repo root or automl/.
automl_dir = Path.cwd() if Path('workflow.py').exists() else Path.cwd() / 'automl'
sys.path.insert(0, str(automl_dir.resolve()))
import pandas as pd
from IPython.display import display, Markdown, FileLink
import workflow as wf

settings = wf.settings()  # Never display this mapping: it may contain credentials.

## 1. Choose your input files

Edit **DATA_FILES** in the next cell. Defaults come from `TRAIN_CSV`, `TEST_CSV`, and `VALIDATION_CSV` in `.env`, or the repository's sample files. You can paste an absolute path, a `~/...` path, or a path relative to the **automl/** directory, regardless of where Jupyter starts.

| Role | What to provide | How it is used |
| --- | --- | --- |
| `train` | Required labeled CSV | Fits the models |
| `test` | Required, separate labeled CSV | Evaluates and ranks candidates; never merged into training |
| `validate` | Optional labeled CSV | Added to training only when `MERGE_VALIDATION = True` |

**Have only your own train and test files?** Replace their paths, set `"validate": None`, and set `MERGE_VALIDATION = False`. Do not merge the repository's validation data with an unrelated training file. AutoML manages its own internal training/validation splits.

Keep all seven feature columns plus `fraud` (0 = not fraud, 1 = fraud), with numeric, finite values and both classes in each supplied split. A CSV with different feature or label names needs an explicit schema change throughout this sample. After changing a setting, rerun this cell and the cells below it. Saved outputs show a previous execution until you rerun them.

In [ ]:
# Edit the values on the right to select your local labeled CSVs.
# Defaults come from .env; relative paths are resolved from automl/.
settings = wf.settings()
DATA_FILES = {
    'train': settings.get('TRAIN_CSV', '../data/train.csv'),
    'test': settings.get('TEST_CSV', '../data/test.csv'),
    'validate': settings.get('VALIDATION_CSV', '../data/validate.csv'),  # None if not supplied.
}
MERGE_VALIDATION = wf.merge_validation_enabled()  # Replace with True or False.
# Example with only two files:
# DATA_FILES = {'train': '/path/to/train.csv', 'test': '/path/to/test.csv', 'validate': None}
# MERGE_VALIDATION = False

# Show fully resolved paths before reading any data.
display(pd.DataFrame([
    {'role': role, 'file': str(wf.data_path(path)) if path else '(not supplied)'}
    for role, path in DATA_FILES.items()
]))

In [3]:
frames = wf.load_data(DATA_FILES)
prepared = wf.prepared_datasets(frames, merge_validation=MERGE_VALIDATION)
source_summary = pd.DataFrame(wf.data_summary(frames))
source_summary['source_file'] = [frames[split].attrs['source_path'] for split in source_summary['split']]
source_summary['purpose'] = source_summary['split'].map({
    'train': 'Training',
    'validate': 'Added to training' if MERGE_VALIDATION else 'Not used in this experiment',
    'test': 'External evaluation only',
})
display(source_summary.style.format({'fraud_rate': '{:.2%}'}))
print('Training data preview:')
display(prepared['train'].head())
print('External test data preview (never used for fitting):')
display(prepared['test'].head())
print('Features:', ', '.join(wf.FEATURES))
print('A majority-class-only classifier would have accuracy:',
      f"{1 - prepared['test']['fraud'].mean():.2%}")

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,75.744071,0.526379,1.178682,1.0,0.0,0.0,1.0,0
1,3.607799,0.687956,0.081721,1.0,1.0,0.0,1.0,0
2,14.303444,8.439479,0.316733,1.0,1.0,0.0,1.0,0
3,113.614272,0.527061,1.748520,1.0,0.0,0.0,1.0,1
4,96.346684,0.448277,1.373413,1.0,0.0,0.0,1.0,0


,split,rows,fraud_rows,fraud_rate
0,train,600000,52535,8.76%
1,validate,200000,17485,8.74%
2,test,200000,17383,8.69%


Features: distance_from_home, distance_from_last_transaction, ratio_to_median_purchase_price, repeat_retailer, used_chip, used_pin_number, online_order
A majority-class-only classifier would have accuracy: 91.31%


## 2. Review exactly what will be uploaded

The table below describes the **two prepared datasets** passed to AutoML, after the optional merge. Both are uploaded with the label column. Test data participates in candidate evaluation/ranking; see the [evaluation caveat](README.md#interpreting-the-results).

The destination is read from your current `.env` / attached Connection. Check the bucket, prefix, and Connection Secret name before running the next section. Credentials are not displayed.

In [4]:
prepared = wf.prepared_datasets(frames, merge_validation=MERGE_VALIDATION)
upload_preview = pd.DataFrame(wf.data_summary(prepared))
upload_preview['purpose'] = ['Fit models', 'Evaluate and rank; never fit']
upload_preview['built_from'] = ['train + validate' if MERGE_VALIDATION else 'train only', 'test only']
display(upload_preview.style.format({'fraud_rate': '{:.2%}'}))
settings = wf.settings()
display(pd.Series({
    'Project namespace': settings.get('RHOAI_PROJECT', '(set RHOAI_PROJECT)'),
    'Connection Secret': wf.required('S3_CONNECTION_NAME'),
    'Input bucket': wf.required('AWS_S3_BUCKET'),
    'Input prefix (a unique suffix will be added)': settings.get('S3_DATA_PREFIX', 'fraud-detection/automl/data'),
    'Pipeline artifact bucket': settings.get('S3_ARTIFACTS_BUCKET') or wf.required('AWS_S3_BUCKET'),
}, name='Upload destination'))
print('Label:', wf.LABEL, '| Binary classification | Positive class: 1')

Merge validation: true
Label: fraud | Binary classification | Positive class: 1


## 3. Upload the prepared train and test CSVs to S3

Run this cell when the preview is correct. Each execution writes a **new** S3 prefix, verifies uploaded sizes, and saves `outputs/data_manifest.json` locally plus a manifest in S3. The manifest records the selected source files, merge decision, and the two uploaded datasets. It does not modify your input files. Existing buckets and Connections are required.

Continue to notebook 01 only after this cell succeeds. Editing paths or merge settings alone does not replace an earlier upload.

In [6]:
s3 = wf.s3_client()
manifest = wf.stage_data(frames, s3, merge_validation=MERGE_VALIDATION)
display(pd.DataFrame(manifest['datasets']).T[['uri', 'rows', 'fraud_rate', 'sha256']])
print('TRAIN dataset:', manifest['datasets']['train']['uri'])
print('TEST dataset (external evaluation):', manifest['datasets']['test']['uri'])
print('Saved state:', wf.OUTPUTS / 'data_manifest.json')

,uri,rows,fraud_rate,sha256
train,s3://YOUR_DATA_BUCKET/fraud-detection/automl/data...,800000,0.087525,84d4b615956c2391739a29a56df3f914749bebaa8544e9...
test,s3://YOUR_DATA_BUCKET/fraud-detection/automl/data...,200000,0.086915,d05f8f71891618474de701f15f45b23bdfb722c9087e12...


Saved state: /path/to/fraud-detection/automl/outputs/data_manifest.json


Next: [01_experiment.ipynb](01_experiment.ipynb)